# Intraday Robust Median Reversion

Tests median-based reversion on hourly BTC-USD with Internal Bar Strength and Hurst features. The download requests the latest 365 days of completed hours. The notebook reports the actual Yahoo Finance coverage used in the backtest.


In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

def get_hourly_data(symbol, start, end):
    prices = yf.download(symbol, start=start, end=end, interval='1h', auto_adjust=False, progress=False, threads=False, multi_level_index=False, timeout=20)
    columns = ['Open', 'High', 'Low', 'Close', 'Volume']
    if prices is None or prices.empty or not set(columns).issubset(prices.columns):
        raise RuntimeError(f'Yahoo Finance returned no hourly OHLCV for {symbol}.')
    bars = prices[columns].dropna(subset=['Open', 'High', 'Low', 'Close']).sort_index()
    bars = bars.loc[~bars.index.duplicated(keep='first')]
    if bars.empty:
        raise RuntimeError(f'{symbol} has no complete hourly OHLC observations.')
    bars = bars.rename(columns=str.lower).rename_axis('date').reset_index(drop=False)
    print(f'Downloaded {len(bars)} hourly bars for {symbol} from Yahoo Finance.')
    print(f"Actual interval {bars['date'].min()} to {bars['date'].max()}")
    return bars

class FeatureEngineer:
    """Calculate lagged returns, Internal Bar Strength, and Hurst features."""

    @staticmethod
    def create_features(df):
        df = df.copy()
        df['log_return'] = np.log(df['close'] / df['close'].shift(1))
        df['log_return'] = df['log_return'].shift(1)
        df['ibs'] = (df['close'] - df['low']) / (df['high'] - df['low'])
        df['ibs'] = df['ibs'].replace([np.inf, -np.inf], np.nan)
        df['ibs'] = df['ibs'].shift(1)
        df['hurst'] = df['close'].rolling(window=100).apply(lambda x: FeatureEngineer._calculate_hurst(x) if len(x) == 100 else np.nan, raw=True)
        df['hurst'] = df['hurst'].shift(1)
        df['target_return'] = np.log(df['close'] / df['close'].shift(1))
        test_idx = ~df[['log_return', 'ibs', 'hurst', 'target_return']].isna().any(axis=1)
        if test_idx.sum() > 100:
            test_data = df[test_idx].tail(500)
            contemp_corr_ret = test_data['log_return'].corr(test_data['target_return'])
            contemp_corr_ibs = test_data['ibs'].corr(test_data['target_return'])
            contemp_corr_hurst = test_data['hurst'].corr(test_data['target_return'])
            print(f'\n🔍 RECENT FEATURE-TO-TARGET CORRELATIONS')
            print(f'  Corr(log_return[t], target[t]): {contemp_corr_ret:.4f}')
            print(f'  Corr(ibs[t], target[t]): {contemp_corr_ibs:.4f}')
            print(f'  Corr(hurst[t], target[t]): {contemp_corr_hurst:.4f}')
        print('\n=== RMR FEATURES ===')
        print('Lagged features:')
        print('  - log_return: return from T-2 to T-1')
        print('  - ibs: Internal Bar Strength T-1')
        print('  - hurst: Hurst exponent through T-1')
        print('Target:')
        print('  - target_return: return from T-1 to T')
        return df

    @staticmethod
    def _calculate_hurst(prices):
        if len(prices) < 20:
            return 0.5
        try:
            log_returns = np.log(prices[1:] / prices[:-1])
            lags = range(2, min(20, len(log_returns) // 2))
            tau = []
            for lag in lags:
                n_subseries = len(log_returns) // lag
                rs_values = []
                for i in range(n_subseries):
                    subseries = log_returns[i * lag:(i + 1) * lag]
                    mean = np.mean(subseries)
                    cumdev = np.cumsum(subseries - mean)
                    R = np.max(cumdev) - np.min(cumdev)
                    S = np.std(subseries)
                    if S > 0:
                        rs_values.append(R / S)
                if len(rs_values) > 0:
                    tau.append(np.mean(rs_values))
            if len(tau) > 1:
                lags_array = np.array(list(lags[:len(tau)]))
                tau_array = np.array(tau)
                valid = (tau_array > 0) & np.isfinite(tau_array)
                if np.sum(valid) > 1:
                    poly = np.polyfit(np.log(lags_array[valid]), np.log(tau_array[valid]), 1)
                    hurst = poly[0]
                    return np.clip(hurst, 0, 1)
            return 0.5
        except:
            return 0.5

class RMR:

    def __init__(self, epsilon=0.5, C=500, lookback=50):
        self.epsilon = epsilon
        self.C = C
        self.lookback = lookback
        self.weights_history = []
        self.hurst_history = []
        self.ibs_history = []
        print(f'\n=== RMR SETTINGS ===')
        print(f'  ε (epsilon): {epsilon} - Reversion threshold')
        print(f'  C: {C} - Maximum aggressiveness')
        print(f'  Lookback: {lookback} - Median window')

    def fit_predict(self, returns, ibs, hurst):
        returns = returns.values
        ibs = ibs.values
        hurst = hurst.values
        n = len(returns)
        weights = np.zeros(n)
        current_weight = 1.0
        rolling_median = pd.Series(returns).rolling(self.lookback).median().shift(1).to_numpy()
        for t in range(n):
            if t < self.lookback:
                weights[t] = current_weight
                continue
            median_return = rolling_median[t]
            last_return = returns[t - 1]
            deviation = last_return - median_return
            loss = current_weight * deviation
            current_hurst = hurst[t - 1] if not np.isnan(hurst[t - 1]) else 0.5
            hurst_factor = 2.0 - 2.0 * current_hurst
            hurst_factor = np.clip(hurst_factor, 0.5, 2.0)
            current_ibs = ibs[t - 1] if not np.isnan(ibs[t - 1]) else 0.5
            ibs_extremeness = abs(current_ibs - 0.5) * 2
            ibs_factor = 1.0 + ibs_extremeness
            adjusted_C = self.C * hurst_factor * ibs_factor
            if abs(loss) > self.epsilon:
                numerator = loss - self.epsilon * np.sign(loss)
                denominator = deviation ** 2
                if denominator > 1e-10:
                    tau = numerator / denominator
                    tau = min(adjusted_C, max(-adjusted_C, tau))
                    current_weight = current_weight - tau * deviation
                    current_weight = max(0, min(2, current_weight))
            weights[t] = current_weight
            self.hurst_history.append(current_hurst)
            self.ibs_history.append(current_ibs)
        self.weights_history = weights
        return weights

    def get_statistics(self):
        weights = np.array(self.weights_history)
        hurst = np.array(self.hurst_history)
        ibs = np.array(self.ibs_history)
        return {'mean_weight': np.mean(weights), 'std_weight': np.std(weights), 'min_weight': np.min(weights), 'max_weight': np.max(weights), 'mean_hurst': np.mean(hurst), 'mean_ibs': np.mean(ibs), 'num_adjustments': np.sum(np.diff(weights) != 0)}

class WalkForwardAnalysis:
    """Evaluate successive time-ordered training, validation, and test blocks."""

    def __init__(self, df, train_size=0.5, validate_size=0.25, n_blocks=4):
        self.df = df.copy()
        self.train_size = train_size
        self.validate_size = validate_size
        self.n_blocks = n_blocks
        self.results = []
        print(f'\n=== WALK-FORWARD CONFIGURATION ===')
        print(f'  Window: expanding')
        print(f'  Initial training observations: {train_size * 100}%')
        print(f'  Validation: {validate_size * 100}%')
        print(f'  Test blocks: {n_blocks}')

    def optimize_hyperparameters(self, train_data, val_data):
        epsilon_grid = [0.1, 0.3, 0.5, 0.7, 1.0]
        C_grid = [100, 300, 500, 700, 1000]
        lookback_grid = [30, 50, 100]
        best_sharpe = -np.inf
        best_params = (0.5, 500, 50)
        context = pd.concat([train_data, val_data])
        print('    Optimizing RMR hyperparameters...')
        for eps in epsilon_grid:
            for c in C_grid:
                for lb in lookback_grid:
                    rmr = RMR(epsilon=eps, C=c, lookback=lb)
                    weights = rmr.fit_predict(context['log_return'], context['ibs'], context['hurst'])
                    val_weights = weights[-len(val_data):]
                    val_returns = val_data['target_return'].values
                    strategy_returns = val_weights * val_returns
                    if len(strategy_returns) > 0 and np.std(strategy_returns) > 0:
                        sharpe = np.mean(strategy_returns) / np.std(strategy_returns) * np.sqrt(365 * 24)
                        if sharpe > best_sharpe:
                            best_sharpe = sharpe
                            best_params = (eps, c, lb)
        print(f'    Selected parameters: ε={best_params[0]}, C={best_params[1]}, lookback={best_params[2]}, Sharpe={best_sharpe:.3f}')
        return (best_params[0], best_params[1], best_params[2], best_sharpe)

    def run(self):
        df_clean = self.df.dropna().reset_index(drop=True)
        if len(df_clean) < 100:
            raise Exception('Insufficient data after removing missing observations')
        print(f'\n✓ Complete observations: {len(df_clean)} records')
        n = len(df_clean)
        initial_train_size = int(n * self.train_size)
        val_size = int(n * self.validate_size)
        remaining = n - initial_train_size - val_size
        test_block_size = remaining // self.n_blocks
        print(f'\n=== WALK-FORWARD STRUCTURE ===')
        print(f'  Initial training observations: {initial_train_size} records')
        print(f'  Validation: {val_size} records')
        print(f'  Test observations per block: {test_block_size} records')
        print(f'  Total blocks: {self.n_blocks}')
        current_train_end = initial_train_size
        for block in range(self.n_blocks):
            if current_train_end + val_size + test_block_size > n:
                print(f'\n⚠ Insufficient data for block {block + 1}, ending the walk-forward test')
                break
            print(f"\n{'=' * 60}")
            print(f'BLOCK {block + 1}/{self.n_blocks}')
            print(f"{'=' * 60}")
            train_start = 0
            train_end = current_train_end
            val_start = train_end
            val_end = val_start + val_size
            test_start = val_end
            test_end = min(test_start + test_block_size, n)
            print(f'  Train: [{train_start}:{train_end}] ({train_end - train_start} records)')
            print(f'  Val:   [{val_start}:{val_end}] ({val_end - val_start} records)')
            print(f'  Test:  [{test_start}:{test_end}] ({test_end - test_start} records)')
            if test_end - test_start < 10:
                print('  ⚠ Skipping undersized block...')
                break
            train_data = df_clean.iloc[train_start:train_end]
            val_data = df_clean.iloc[val_start:val_end]
            test_data = df_clean.iloc[test_start:test_end]
            best_eps, best_C, best_lb, val_sharpe = self.optimize_hyperparameters(train_data, val_data)
            rmr = RMR(epsilon=best_eps, C=best_C, lookback=best_lb)
            context = pd.concat([train_data, val_data, test_data])
            all_weights = rmr.fit_predict(context['log_return'], context['ibs'], context['hurst'])
            test_weights = all_weights[-len(test_data):]
            test_returns = test_data['target_return'].values
            strategy_returns = test_weights * test_returns
            rmr_stats = {'mean_hurst': np.mean(rmr.hurst_history[-len(test_data):]),
                         'mean_ibs': np.mean(rmr.ibs_history[-len(test_data):])}
            if len(strategy_returns) > 0 and np.std(strategy_returns) > 0:
                test_sharpe = np.mean(strategy_returns) / np.std(strategy_returns) * np.sqrt(365 * 24)
                total_return = np.exp(np.sum(strategy_returns)) - 1
                max_dd = self.calculate_max_drawdown(strategy_returns)
            else:
                test_sharpe = 0
                total_return = 0
                max_dd = 0
            print(f'\n  === BLOCK RESULTS {block + 1} ===')
            print(f'    Sharpe (validation): {val_sharpe:.3f}')
            print(f'    Sharpe (test): {test_sharpe:.3f}')
            print(f'    Total Return (test): {total_return * 100:.2f}%')
            print(f'    Max Drawdown (test): {max_dd * 100:.2f}%')
            print(f"    Mean Hurst exponent: {rmr_stats['mean_hurst']:.3f}")
            print(f"    Mean IBS: {rmr_stats['mean_ibs']:.3f}")
            self.results.append({'block': block + 1, 'train_size': train_end - train_start, 'test_size': test_end - test_start, 'epsilon': best_eps, 'C': best_C, 'lookback': best_lb, 'sharpe_validation': val_sharpe, 'sharpe_test': test_sharpe, 'total_return_test': total_return, 'max_drawdown_test': max_dd, 'mean_hurst': rmr_stats['mean_hurst'], 'mean_ibs': rmr_stats['mean_ibs'], 'test_start_date': df_clean.iloc[test_start]['date'], 'test_end_date': df_clean.iloc[test_end - 1]['date']})
            current_train_end += test_block_size
        return self.results

    @staticmethod
    def calculate_max_drawdown(returns):
        cumulative = np.exp(np.cumsum(returns))
        running_max = np.maximum.accumulate(cumulative)
        drawdown = (cumulative - running_max) / running_max
        return np.min(drawdown)

def main():
    """Run the hourly RMR walk-forward backtest."""
    print('\n' + '=' * 60)
    print('RMR FRAMEWORK FOR HOURLY BTC-USD')
    print('=' * 60)
    print('\nObjective: robust median mean reversion')
    print('FEATURES: log_return, IBS, Hurst exponent')
    print('Technical features use lagged observations.')
    print('=' * 60)
    symbol = 'BTC-USD'
    end = pd.Timestamp.now(tz='UTC').floor('h')
    start = end - pd.Timedelta(days=365)
    try:
        df = get_hourly_data(symbol, start, end)
        engineer = FeatureEngineer()
        df = engineer.create_features(df)
        print(f'\n✓ Dataset: {len(df)} records')
        print(f"  NaNs: log_return={df['log_return'].isna().sum()}, ibs={df['ibs'].isna().sum()}, hurst={df['hurst'].isna().sum()}")
        wfa = WalkForwardAnalysis(df=df, train_size=0.6, validate_size=0.2, n_blocks=3)
        results = wfa.run()
        print('\n' + '=' * 60)
        print('WALK-FORWARD SUMMARY')
        print('=' * 60)
        if len(results) > 0:
            results_df = pd.DataFrame(results)
            print(results_df[['block', 'epsilon', 'C', 'lookback', 'sharpe_test', 'total_return_test', 'mean_hurst', 'mean_ibs']])
            avg_sharpe = results_df['sharpe_test'].mean()
            avg_hurst = results_df['mean_hurst'].mean()
            print(f'\n=== AGGREGATE METRICS ===')
            print(f'  Mean Sharpe ratio: {avg_sharpe:.3f}')
            print(f"  Mean Hurst exponent: {avg_hurst:.3f} {('(mean-reverting)' if avg_hurst < 0.5 else '(trending)')}")
            print(f'  Blocks: {len(results)}')
        print('\n' + '=' * 60)
        print('MODEL CHECKS')
        print('=' * 60)
        print('✓ Lagged technical features')
        print('✓ RMR: Median-based estimator')
        print('✓ Adaptive sizing uses Hurst and IBS')
        print('=' * 60)
    except Exception as e:
        print(f'\n❌ ERROR: {str(e)}')
        import traceback
        traceback.print_exc()
        raise
if __name__ == '__main__':
    main()
